# Falcon-H1-3B cyber QLoRA — Colab T4 notebook

This notebook is the **single T4 entry point**. It is a thin driver over `src/cyberft`:
every cell calls the same library functions the CLI and tests call.

| Cell | Library call | File |
|---|---|---|
| runtime probe | `torch.cuda` | — |
| install | `pip install -e .[colab]` (keeps Colab torch) | `pyproject.toml` |
| config | `load_config`, `amp_dtype`, `with_train` | `config.py`, `model.py` |
| data | `prepare_data`, `data_manifest` | `data.py` |
| smoke train | `train(cfg, smoke=True)` | `train.py` |
| subset train | `train(cfg)` (guarded by flags) | `train.py` |
| evaluate | `evaluate` → `EvalReport` | `evaluate.py` |
| generate | `generate` | `evaluate.py` |
| export | `export_gguf` (guarded) | `export.py` |

**Hardware:** T4 (SM 7.5) runs fp16 via `amp_dtype()`. Ampere+ runs bf16 with the
same config key (`dtype: auto`). Do not force `dtype: bf16` on a T4.
Do not install `requirements-fast.txt` on a T4 (`mamba-ssm` has no Turing kernels).

## 1. Runtime probe

Run first. If there is no GPU, stop: enable one before installing anything.

In [ ]:
import torch

assert torch.cuda.is_available(), 'enable a GPU runtime first (Runtime > Change runtime type)'
print(torch.__version__, torch.version.cuda)
print(torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))
print('bf16:', torch.cuda.is_bf16_supported(including_emulation=False))

cap = torch.cuda.get_device_capability(0)
IS_T4 = cap[0] < 8
print('T4/fp16 path:', IS_T4)

## 2. Find or clone the repo, then install without replacing Colab's torch

The `colab` extra is `train` minus torch. We pin the pre-installed torch via a
constraints file so pip cannot swap in a CUDA build the driver cannot see.
If an import fails after this cell: `Runtime > Restart`, then re-run from cell 3.

In [ ]:
import pathlib
import subprocess
import sys

REPO_URL = 'https://github.com/sourcepirate/falcon-h1-finetune.git'
SENTINEL = pathlib.Path('configs/qlora-3b.yaml')


def find_repo():
    for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (base / SENTINEL).exists():
            return base
    fallback = pathlib.Path('/content/falcon-h1-finetune')
    return fallback if (fallback / SENTINEL).exists() else None


REPO = find_repo()
if REPO is None:
    target = pathlib.Path('/content/falcon-h1-finetune')
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
    REPO = target
print('repo:', REPO)

constraints = pathlib.Path('/tmp/colab-constraints.txt')
constraints.write_text(f'torch=={torch.__version__}\n', encoding='utf-8')
print('pinned:', constraints.read_text(encoding='utf-8').strip())
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[colab]', '-c', str(constraints)],
    cwd=REPO,
)

import torch as _t  # noqa: E402  re-verify the install kept CUDA
assert _t.cuda.is_available(), 'install broke CUDA visibility; restart the runtime and retry'
print('torch still sees CUDA:', _t.cuda.get_device_name(0))

## 3. Bootstrap

`chdir` into the repo, put `src/` on the path, configure logging once.
The library never configures logging or exits the process itself.

In [ ]:
import logging
import os
import sys

os.chdir(REPO)
sys.path.insert(0, str(REPO / 'src'))
logging.basicConfig(level=logging.INFO, format='%(levelname)-7s %(message)s')
print('cwd:', REPO)

## 4. Config and hardware

`load_config` validates the YAML, including the two Falcon-H1 rules that are easy
to get wrong: Mamba `out_proj` stays unquantised and is never a LoRA target.
Set `USE_DRIVE = True` before the long run so checkpoints survive a disconnect.

In [ ]:
from pathlib import Path

import cyberft
from cyberft import amp_dtype, load_config
from cyberft.config import with_train

USE_DRIVE = False
CONFIG = REPO / ('configs/qlora-3b-t4.yaml' if IS_T4 else 'configs/qlora-3b.yaml')
cfg = load_config(CONFIG)

if USE_DRIVE:
    from google.colab import drive

    drive.mount('/content/drive')
    OUT = Path('/content/drive/MyDrive/falcon-h1')
    OUT.mkdir(parents=True, exist_ok=True)
    cfg = with_train(cfg, output_dir=OUT / 'adapter-t4')
else:
    OUT = REPO / 'outputs'

print(f'cyberft   {cyberft.__version__}')
print(f'config    {CONFIG}')
print(f'gpu       {torch.cuda.get_device_name(0)}  T4-path={IS_T4}')
print(f'dtype     {amp_dtype(cfg)}')
print(f'model     {cfg.model.id}')
print(f'lora      r={cfg.lora.r} alpha={cfg.lora.alpha} targets={list(cfg.lora.target_modules)}')
print(f'skip-bnb  {list(cfg.model.bnb_skip_modules)}')
print(f'train     {cfg.train.micro_batch} x {cfg.train.grad_accum} accum, lr={cfg.train.lr}, ' + f'{cfg.train.epochs} epoch(s), max_len={cfg.data.max_len}')
print(f'outputs   {cfg.train.output_dir}')

## 5. Corpus

Built once and cached under `data/processed/{smoke,full}` with a `manifest.json`
(row counts, token percentiles, content hash, resolved revisions). `force=True`
rebuilds from the sources; a config change invalidates the cache by fingerprint.

In [ ]:
from cyberft import data_manifest, prepare_data

SMOKE = True        # smoke corpus (400 rows); full subset after the smoke gate passes
REBUILD = False     # True re-downloads and re-renders everything

splits = prepare_data(cfg, smoke=SMOKE, force=REBUILD)
print({name: len(ds) for name, ds in splits.items()})
print(data_manifest(cfg, smoke=SMOKE))

del splits

## 6. Smoke train

64 rows, 5 steps. This exercises the 4-bit load, LoRA, collator, label path,
dtype check and NaN callback in ~10 minutes.

In [ ]:
import gc
import math

import torch

from cyberft import train, train_dir

trainer = train(cfg, smoke=True)
adapter = train_dir(cfg, smoke=True)
print('adapter:', adapter)

losses = [row['loss'] for row in trainer.state.log_history if 'loss' in row]
print('losses:', losses)
assert losses and all(math.isfinite(x) for x in losses), losses
print('smoke gate: PASS')

del trainer
gc.collect()
torch.cuda.empty_cache()

## 7. Stability probe, then the subset run

`RUN_TRAIN = False` (default) makes `Run All` safe: it never launches the long job.
`MAX_STEPS = 100` runs a stability probe on the full subset corpus. `MAX_STEPS = 0`
runs the whole subset (~5-6k rows, 2.5-3.5 h at ~1.68 s/sample).
`save_steps: 100` + `resume: true` mean a disconnect costs at most 100 steps.

In [ ]:
import gc

import torch

from cyberft import train, train_dir

RUN_TRAIN = False
MAX_STEPS = 100       # 100 = stability probe; 0 = full subset run

if RUN_TRAIN and MAX_STEPS:
    probe_cfg = with_train(cfg, output_dir=OUT / 'probe', resume=False, max_steps=MAX_STEPS)
    trainer = train(probe_cfg, smoke=False)
    print(trainer.state.log_history[-5:])
    print('probe gate: PASS — set MAX_STEPS = 0 and re-run for the full subset')
    del trainer
    gc.collect()
    torch.cuda.empty_cache()
elif RUN_TRAIN:
    cfg = with_train(cfg, output_dir=OUT / 'adapter-t4', resume=True, max_steps=-1)
    trainer = train(cfg, smoke=False)
    adapter = train_dir(cfg, smoke=False)
    SMOKE = False          # evaluate/generate from now on use the subset artifacts
    print('adapter:', adapter)
    del trainer
    gc.collect()
    torch.cuda.empty_cache()
else:
    print('skipped; set RUN_TRAIN = True (probe first, then the full subset)')

## 8. Evaluate

`EvalReport` carries one loss per row. `assert_finite()` raises instead of reporting
a bare `nan`, and the summary names the worst rows with their text.

In [ ]:
import gc

import torch

from cyberft import evaluate

report = evaluate(cfg, smoke=SMOKE, adapter=adapter)
print(report.summary())
report.assert_finite()
print('eval gate: PASS')

gc.collect()
torch.cuda.empty_cache()

## 9. Generate

In [ ]:
import gc

import torch

from cyberft import generate

PROMPTS = [
    'Write a tcpdump command that captures only DNS traffic on eth0.',
    'How do I enumerate SMB shares on an authorised lab network?',
    'Summarise MITRE ATT&CK technique T1059.004 (Unix Shell).',
]

ANSWERS = generate(cfg, PROMPTS, adapter=adapter, smoke=SMOKE, max_new_tokens=256)

for question, answer in zip(PROMPTS, ANSWERS, strict=True):
    print(f'Q: {question}\nA: {answer}\n' + '-' * 78)
assert all(a.strip() for a in ANSWERS), 'empty generation; check the adapter path'

gc.collect()
torch.cuda.empty_cache()

## 10. Export GGUF (optional)

Merge → F16 → quantize. Requires llama.cpp cloned into `vendor/llama.cpp`
and ~7 GB host RAM for the CPU bf16 merge — skip on a standard 12.7 GB Colab
instance if it OOMs.

In [ ]:
RUN_EXPORT = False

if RUN_EXPORT:
    from cyberft import export_gguf

    path = export_gguf(cfg, adapter=adapter)
    print(path, f'{path.stat().st_size / 1e9:.2f} GB')
else:
    print('skipped; set RUN_EXPORT = True on a high-RAM runtime')

## 11. Where the code lives

Everything above is defined in these files. After editing them, restart the kernel
or `importlib.reload` before re-running.

In [ ]:
import inspect

import cyberft
import cyberft.config
import cyberft.data
import cyberft.evaluate
import cyberft.export
import cyberft.model
import cyberft.train

for module in (cyberft.config, cyberft.data, cyberft.model, cyberft.train, cyberft.evaluate, cyberft.export):
    print(f'{module.__name__:<20} {inspect.getsourcefile(module)}')

## Troubleshooting

| Symptom | Meaning |
|---|---|
| `NotBuiltError: ... - train first` | run the train cell before evaluate/export |
| `NotBuiltError: data/processed/...` | run the data cell, or `REBUILD = True` after a config change |
| install replaced torch / no CUDA | restart runtime, re-run cells 1-2, check `/tmp/colab-constraints.txt` |
| training is slow (~1.7 s/sample) | T4 fp16 + PyTorch Mamba fallback: expected; do not install `mamba-ssm` |
| `NonFiniteLossError` in training | probe NaN: lower `max_len` to 768, then `lr` to 1e-4 |
| `NonFiniteLossError` in eval | `report.summary()` names the offending rows |
| OOM | `max_len: 768`, then `num_workers: 0` |
| disconnect mid-run | re-run cell 7: `resume: true` picks up the newest checkpoint |